In [21]:
// ★案Ａ★　総当たり
use std::fs::File;
use std::io::{Write, BufWriter};
use std::time::Instant;

// 1. 設問文 (&str型)
let ciphertext = "EBG KVVV vf n fvzcyr yrggre fhofgvghgvba pvcure gung ercynprf n yrggre jvgu gur yrggre KVVV yrggref nsgre vg va gur nycunorg. EBG KVVV vf na rknzcyr bs gur Pnrfne pvcure, qrirybcrq va napvrag Ebzr. Synt vf SYNTFjmtkOWFNZdjkkNH. Vafreg na haqrefpber vzzrqvngryl nsgre SYNT.";

// 2. 文字シフト用関数 
fn shift_char(c: char, n: usize) -> char {
    let n = n % 26; // 26回で一周するので
    if c.is_ascii_uppercase() {
        // A(65) ～ Z(90) の間で回転
        let base = 'A' as u8;
        (((c as u8 - base) + n as u8) % 26 + base) as char
    } else if c.is_ascii_lowercase() {
        // a(97) ～ z(122) の間で回転
        let base = 'a' as u8;
        (((c as u8 - base) + n as u8) % 26 + base) as char
    } else {
        c // 記号や空白はそのまま
    }
}

// 3. 処理開始
let start_time = Instant::now();
let file = File::create("result.txt").expect("ファイルが作れませんでした");
let mut writer = BufWriter::new(file);

println!("解析を開始します...");

for shift in 1..52 {
    // 一文字ずつ変換して新しいStringを作る
    let decrypted: String = ciphertext.chars().map(|c| shift_char(c, shift)).collect();
    
    // FLAGが含まれているかチェック
    let has_flag = decrypted.contains("FLAG");
    
    // 経過時間を計算
    let elapsed = start_time.elapsed().as_micros(); // マイクロ秒単位

    // ファイルに詳細を書き込む
    writeln!(
        writer, 
        "Shift {:02} | Found FLAG: {:5} | Time: {:10}μs | Text: {}", 
        shift, has_flag, elapsed, decrypted
    ).unwrap();

    if has_flag {
        println!("✨ Shift {:02} で FLAG を検出しました！ (詳細は result.txt へ)", shift);
    }
}

println!("すべての試行が完了しました。総処理時間: {:?}", start_time.elapsed());

解析を開始します...
✨ Shift 13 で FLAG を検出しました！ (詳細は result.txt へ)
✨ Shift 39 で FLAG を検出しました！ (詳細は result.txt へ)
すべての試行が完了しました。総処理時間: 1.4565ms


In [22]:
// ★案Ｂ★　差分シグネチャ分析
use std::time::Instant;

// 1. 設問文の定義
let ciphertext = "EBG KVVV vf n fvzcyr yrggre fhofgvghgvba pvcure gung ercynprf n yrggre jvgu gur yrggre KVVV yrggref nsgre vg va gur nycunorg. EBG KVVV vf na rknzcyr bs gur Pnrfne pvcure, qrirybcrq va napvrag Ebzr. Synt vf SYNTFjmtkOWFNZdjkkNH. Vafreg na haqrefpber vzzrqvngryl nsgre SYNT.";

let start = Instant::now();

// --- 工程1: 数値変換 ---
let sequence: Vec<i32> = ciphertext
    .chars()
    .filter(|c| c.is_ascii_alphabetic())
    .map(|c| {
        if c.is_ascii_uppercase() {
            (c as i32) - ('A' as i32)
        } else {
            (c as i32) - ('a' as i32)
        }
    })
    .collect();

println!("工程1完了: {} 文字のアルファベットを抽出しました。", sequence.len());

// --- 工程2: 差分計算 ---
let diffs: Vec<i32> = sequence
    .windows(2)
    .map(|w| (w[1] - w[0] + 26) % 26)
    .collect();

println!("工程2完了: {} 個の差分データを生成しました。", diffs.len());

// --- 工程3: 検索 ---
let target_pattern = vec![6, 15, 6]; // F->L, L->A, A->G の差分
let mut found_count = 0;

for (i, window) in diffs.windows(target_pattern.len()).enumerate() {
    if window == target_pattern {
        let original_char_idx = sequence[i];
        let shift = (original_char_idx - 5 + 26) % 26; // 'F'は5番目(0始まり)
        println!("✨ ターゲット検出! インデックス: {}, シフト回数: {}", i, shift);
        found_count += 1;
    }
}

println!("工程3完了: {} 件のFLAG候補が見つかりました。", found_count);
println!("総処理時間: {:?}", start.elapsed());

工程1完了: 224 文字のアルファベットを抽出しました。
工程2完了: 223 個の差分データを生成しました。
✨ ターゲット検出! インデックス: 160, シフト回数: 13
✨ ターゲット検出! インデックス: 166, シフト回数: 13
✨ ターゲット検出! インデックス: 220, シフト回数: 13
工程3完了: 3 件のFLAG候補が見つかりました。
総処理時間: 64.7µs


In [23]:
// ★案Ｃ★　統計的手法
use std::collections::HashMap;
use std::time::Instant;

// 1. 設問文
let ciphertext = "EBG KVVV vf n fvzcyr yrggre fhofgvghgvba pvcure gung ercynprf n yrggre jvgu gur yrggre KVVV yrggref nsgre vg va gur nycunorg. EBG KVVV vf na rknzcyr bs gur Pnrfne pvcure, qrirybcrq va napvrag Ebzr. Synt vf SYNTFjmtkOWFNZdjkkNH. Vafreg na haqrefpber vzzrqvngryl nsgre SYNT.";

let start = Instant::now();

// --- 工程1: 文字の出現頻度を数える ---
let mut counts = HashMap::new();
for c in ciphertext.chars().filter(|c| c.is_ascii_alphabetic()) {
    // 大文字・小文字を区別せずにカウント
    let lower_c = c.to_ascii_lowercase();
    *counts.entry(lower_c).or_insert(0) += 1;
}

// --- 工程2: 最も頻出する文字を特定する ---
let most_frequent_char = counts
    .iter()
    .max_by_key(|&(_, count)| count)
    .map(|(&c, _)| c)
    .expect("英字が見つかりませんでした");

println!("最も頻出する文字: '{}'", most_frequent_char);

// --- 工程3: 'e' との差分からシフト回数を推測 ---
// a=0, b=1 ... z=25
let most_frequent_idx = (most_frequent_char as i32) - ('a' as i32);
let e_idx = ('e' as i32) - ('a' as i32);

// シフト回数 = (頻出文字 - 'e')
let estimated_shift = (most_frequent_idx - e_idx + 26) % 26;

println!("'{}' が 'e' だと仮定した場合のシフト回数: {}", most_frequent_char, estimated_shift);

// --- 工程4: 推測した回数で実際にFLAGがあるか確認 ---
// (案Aのロジックを1回だけ実行する)
fn shift_char(c: char, n: i32) -> char {
    if !c.is_ascii_alphabetic() { return c; }
    let base = if c.is_ascii_uppercase() { 'A' } else { 'a' } as i32;
    (((c as i32 - base) + n) % 26 + base) as u8 as char
}

let decrypted: String = ciphertext.chars().map(|c| shift_char(c, estimated_shift)).collect();

if decrypted.contains("FLAG") {
    println!("✨ 推測的中！正解の英文を確認しました。");
} else {
    println!("残念、推測が外れました。");
}

println!("処理時間: {:?}", start.elapsed());

最も頻出する文字: 'r'
'r' が 'e' だと仮定した場合のシフト回数: 13
✨ 推測的中！正解の英文を確認しました。
処理時間: 50.8µs


In [24]:
// ★おまけ★
use std::fs::File;
use std::io::Write;

{
    // r##" を使うことで、中の "#ff0000" などの記号と干渉しなくなります
    let html = r##"
<!DOCTYPE html>
<html>
<head><meta charset="UTF-8"><title>Decryption</title></head>
<body style="background:#111;color:#0f0;font-family:monospace;display:flex;justify-content:center;padding:20px;">
    <div style="border:1px solid #444;padding:20px;background:#222;width:600px;box-shadow:0 0 20px #000;">
        <h3 id="stat" style="color:#fa0;">SCANNING...</h3>
        <input type="range" id="sld" min="0" max="25" value="0" style="width:100%;">
        <div style="margin:10px 0;">SHIFT: <span id="sv">0</span></div>
        <div id="out" style="background:#000;padding:15px;min-height:120px;border:1px solid #333;word-break:break-all;"></div>
    </div>
    <script>
        const src = "EBG KVVV vf n fvzcyr yrggre fhofgvghgvba pvcure gung ercynprf n yrggre jvgu gur yrggre KVVV yrggref nsgre vg va gur nycunorg. EBG KVVV vf na rknzcyr bs gur Pnrfne pvcure, qrirybcrq va napvrag Ebzr. Synt vf SYNTFjmtkOWFNZdjkkNH. Vafreg na haqrefpber vzzrqvngryl nsgre SYNT.";
        const sld = document.getElementById('sld');
        const sv = document.getElementById('sv');
        const out = document.getElementById('out');
        const stat = document.getElementById('stat');
        function shift(c, n) {
            const k = c.charCodeAt(0);
            if (k >= 65 && k <= 90) return String.fromCharCode((k - 65 + n) % 26 + 65);
            if (k >= 97 && k <= 122) return String.fromCharCode((k - 97 + n) % 26 + 97);
            return c;
        }
        function draw(n) {
            sv.innerText = n;
            let res = "";
            for (let i = 0; i < src.length; i++) res += shift(src[i], n);
            if (res.indexOf("FLAG") !== -1) {
                stat.innerText = "DETECTED!"; stat.style.color = "#0f0";
                out.innerHTML = res.replace(/(FLAG)/g, '<span style="background:yellow;color:red;font-weight:bold;">$1</span>');
            } else {
                out.innerText = res;
            }
        }
        let n = 0, auto = true;
        const timer = setInterval(() => {
            if (!auto) return;
            n = (n + 1) % 26;
            sld.value = n;
            draw(n);
        }, 500);
        sld.oninput = () => { auto = false; stat.innerText = "MANUAL"; stat.style.color = "#0af"; draw(parseInt(sld.value)); };
        draw(0);
    </script>
</body>
</html>
"##;

    let mut f = File::create("auto_cipher_tool.html").expect("err");
    f.write_all(html.as_bytes()).expect("err");
    println!("✅ DONE: auto_cipher_tool.html");
}

✅ DONE: auto_cipher_tool.html


()

### 考察：設問設計に関する仮説
今回の測定結果（案Cにおいて 'e' が最頻値となる挙動）から、出題者の意図について以下の仮説を立てた。

* **仮説：案Cが本来想定されていた設計である**
    * 理由1：実測データにおいて 'e' が顕著に現れる設計になっている。
    * 理由2：設問文中に「FLAG」以外の文字が含まれている点。
    * 結論：もし「FLAG」だけが重要であれば他の文字を出す必要はない。あえて他の文字を含ませているのは、この案Cのような解析プロセス（あるいは特定の文字への偏り）を導き出すための意図的な設計であると感じられる。